# Customer Segmentation Analysis

This project performs customer segmentation using RFM analysis and K-means clustering on retail transaction data.

## Objectives
- Clean and prepare transactional data
- Create RFM (Recency, Frequency, Monetary) features
- Segment customers using clustering
- Generate business insights for targeting and retention

In [ ]:
#Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore")

In [ ]:
#Load Data
df = pd.read_excel("data/raw/Online Retail.xlsx")

print("Shape:", df.shape)
df.head()

In [ ]:
#Clean Data
# Remove missing customers
df = df.dropna(subset=["CustomerID"])

# Remove returns
df = df[df["Quantity"] > 0]

# Remove invalid prices
df = df[df["UnitPrice"] > 0]

# Convert types
df["CustomerID"] = df["CustomerID"].astype(int)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Create total price
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

print("Cleaned shape:", df.shape)

In [ ]:

#RFM Features
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    "InvoiceNo": "nunique",
    "TotalPrice": "sum"
}).reset_index()

rfm.columns = ["CustomerID", "Recency", "Frequency", "Monetary"]

rfm.head()

In [ ]:
#Scale Data
features = rfm[["Recency", "Frequency", "Monetary"]]

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(features)

In [ ]:
#K-Means Clustering
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

rfm.head()

In [ ]:
#Label Segments
cluster_summary = rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].mean()

# Rank clusters
cluster_summary["Score"] = (
    -cluster_summary["Recency"] +
    cluster_summary["Frequency"] +
    cluster_summary["Monetary"] / 100
)

cluster_summary = cluster_summary.sort_values("Score", ascending=False)

labels = ["VIP", "Loyal", "Potential", "At-Risk"]

cluster_summary["Segment"] = labels

cluster_map = cluster_summary["Segment"].to_dict()

rfm["Segment"] = rfm["Cluster"].map(cluster_map)

rfm.head()

In [ ]:
#Segment Summary
segment_summary = rfm.groupby("Segment").agg({
    "CustomerID": "count",
    "Monetary": "sum"
}).reset_index()

segment_summary.columns = ["Segment", "Customers", "Revenue"]

segment_summary

In [ ]:
#Visuals
#Revenue by segment
segment_summary.sort_values("Revenue", ascending=False).plot(
    x="Segment", y="Revenue", kind="bar", legend=False
)

plt.title("Revenue by Segment")
plt.ylabel("Revenue")
plt.xticks(rotation=20)
plt.show()

In [ ]:
#Save Files
rfm.to_csv("data/processed/rfm_segments.csv", index=False)
segment_summary.to_csv("data/processed/segment_summary.csv", index=False)

print("Saved processed files.")

## Key Insights

- VIP customers generate the highest revenue and should be prioritized for retention.
- Loyal customers purchase frequently and are strong candidates for upselling.
- Potential customers show moderate engagement and can be nurtured with targeted campaigns.
- At-risk customers have low engagement and may require win-back strategies.

## Business Impact
This segmentation enables more effective targeting, improved retention strategies, and increased revenue through data-driven decision-making.